In [0]:
dbutils.widgets.text("catalog", "bank_risk_dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.governance")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.governance.mask_pii(v STRING)
RETURNS STRING
COMMENT 'Muestra el valor completo solo a risk_team; a los demás les deja ver los últimos 4 caracteres'
RETURN CASE
  WHEN v IS NULL THEN NULL
  WHEN is_account_group_member('risk_team') THEN v
  ELSE concat(repeat('*', greatest(length(v) - 4, 0)), right(v, 4))
END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.governance.mask_name(v STRING)
RETURNS STRING
COMMENT 'Muestra el nombre completo solo a risk_team; a los demás solo la inicial'
RETURN CASE
  WHEN v IS NULL THEN NULL
  WHEN is_account_group_member('risk_team') THEN v
  ELSE concat(substr(v, 1, 1), '*****')
END
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.governance.alert_filter(rule_code STRING)
RETURNS BOOLEAN
COMMENT 'risk_team ve todas las alertas; fraud_analysts solo las de reglas de fraude. Las alertas PLD son confidenciales'
RETURN is_account_group_member('risk_team')
  OR (is_account_group_member('fraud_analysts')
      AND rule_code IN ('R02_BENEF_DISPOSITIVO_NUEVO', 'R03_NOCTURNO'))
""")

In [0]:
display(spark.sql(f"""
SELECT
  {catalog}.governance.mask_pii('RASJ7801104CB') AS rfc,
  {catalog}.governance.mask_pii('999123000045678912') AS clabe,
  {catalog}.governance.mask_name('José Ramírez Sánchez') AS nombre,
  {catalog}.governance.alert_filter('R04_FRACCIONAMIENTO') AS ve_alerta_pld,
  is_account_group_member('risk_team') AS soy_risk_team
"""))

In [0]:
access = {
    "credit_analysts": ["dim_customer", "dim_loan", "dim_date", "fact_loan_month", "mv_credit_risk"],
    "fraud_analysts": ["dim_customer", "dim_account", "dim_bank", "dim_rule", "dim_date",
                       "fact_transfer", "fact_alert", "mv_fraud", "mv_alerts"],
}

for group in ["credit_analysts", "fraud_analysts", "risk_team"]:
    spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `{group}`")
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.gold TO `{group}`")

spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.gold TO `risk_team`")

for group, tables in access.items():
    for t in tables:
        spark.sql(f"GRANT SELECT ON TABLE {catalog}.gold.{t} TO `{group}`")

In [0]:
display(spark.sql(f"""
SELECT table_name, column_name, mask_name
FROM {catalog}.information_schema.column_masks
WHERE table_schema = 'gold'
"""))

display(spark.sql(f"""
SELECT table_name, filter_name, target_columns
FROM {catalog}.information_schema.row_filters
WHERE table_schema = 'gold'
"""))

display(spark.sql(f"SHOW GRANTS ON TABLE {catalog}.gold.fact_alert"))

In [0]:
display(spark.sql(f"""
SELECT customer_id, full_name, rfc, curp
FROM {catalog}.gold.dim_customer
LIMIT 5
"""))

display(spark.sql(f"""
SELECT r.rule_type, COUNT(*) AS alertas
FROM {catalog}.gold.fact_alert a
JOIN {catalog}.gold.dim_rule r ON a.rule_code = r.rule_code
GROUP BY r.rule_type
"""))